## Imports and definitions

In [ ]:
from ddsp import DDSP, AudioDataset
from ddsp.callbacks import BetaWarmupEpochCallback
from ddsp.synths import NoiseBandSynth, SineSynth
from ddsp.utils import find_checkpoint

from lightning.pytorch.callbacks import ModelCheckpoint
from lightning import Trainer

from IPython.display import Audio, display

from torch.utils.data import DataLoader, Subset, random_split

import torch
import umap

torch.set_float32_matmul_precision('medium')
torch.set_default_tensor_type('torch.cuda.FloatTensor')
torch.set_default_device('cuda')

import os

training_root = '/home/btadeusz/code/ddsp_vae/training/experiments/fitting_multiple_datasets/'

In [ ]:
# Dataset parameters
chunk_duration = 2.0
sampling_rate = 44100
n_signal = chunk_duration * sampling_rate
batch_size = 8

# Model parameters
latent_size = num_params = 4
max_freq = 22050
n_filters = 500
n_sines = 100

# Training parameters
warmup_start = 300
warmup_end = 500
beta = 1.0
max_epochs = 1000
learning_rate = 1e-4


In [ ]:
def get_dataset_split(dataset_path, validation_split=0.2):
  """
  Splits the dataset into training and validation sets.
  """
  generator=torch.Generator(device='cuda')

  dataset_A = AudioDataset(dataset_path=dataset_path, n_signal=n_signal)
  total_len = len(dataset_A)

  val_len = int(validation_split * total_len)  # 20% for validation
  indices = torch.randperm(total_len, generator=generator)

  val_indices = indices[:val_len]
  train_indices = indices[val_len:]

  train_set = Subset(dataset_A, train_indices)
  val_set = Subset(dataset_A, val_indices)

  train_loader = DataLoader(train_set, batch_size, shuffle=True, num_workers=0, generator=generator)
  val_loader = DataLoader(val_set, batch_size, shuffle=False, num_workers=0, generator=generator)

  return train_loader, val_loader


def build_ddsp_model():
  """
  Builds the DDSP model with the specified configurations.
  """
  nbn = NoiseBandSynth.to_config(
    n_filters=n_filters,
    fs=sampling_rate,
  )

  sines = SineSynth.to_config(
    n_sines=n_sines,
    fs=sampling_rate,
  )

  ddsp = DDSP(
    synth_configs=[nbn, sines],
    fs=sampling_rate,
    latent_size=latent_size,
    num_params=num_params,
    learning_rate=learning_rate,
    perceptual_loss_weight=0,
    plateau_patience=200,
  ).to('cuda')

  return ddsp


def build_trainer(model_training_path):
  training_callbacks = []
  beta_warmup = BetaWarmupEpochCallback(
    beta=beta,
    start_epoch=warmup_start,
    end_epoch=warmup_end
  )
  training_callbacks.append(beta_warmup)

  best_checkpoint_callback = ModelCheckpoint(
    filename='best',
    monitor='val_loss',
    mode='min',
    save_top_k=1,
    save_last=True,
    dirpath=model_training_path
  )
  training_callbacks.append(best_checkpoint_callback)

  trainer = Trainer(
    callbacks=training_callbacks,
    max_epochs=max_epochs,
    accelerator='cuda',
    precision=16,
  )

  return trainer


def train_on_dataset(dataset_path, model_name=None):
  """
  Trains the DDSP model on a specific dataset.
  """
  train_loader, val_loader = get_dataset_split(dataset_path)

  model = build_ddsp_model()

  model_training_path = os.path.join(training_root, model_name)
  trainer = build_trainer(model_training_path)

  ckpt = find_checkpoint(model_training_path, return_none=True, typ='last')
  print("Found ckpt: ", ckpt)
  trainer.fit(model, train_loader, val_loader, ckpt_path=ckpt)

  return model, trainer, train_loader, val_loader


def examine_model(model, val_loader):
  """
  Examines the trained model by displaying original and autoencoded audio samples,
  as well as reporting on the loss.
  """
  model = model.cuda()
  model.eval()
  with torch.no_grad():
    x_audio = next(iter(val_loader))
    x_audio = x_audio.to('cuda')

    # Forward pass through the model
    y_audio = model(x_audio).squeeze(1)

    print(x_audio.shape, y_audio.shape)

    # Display original and reconstructed audio
    for j in range(x_audio.shape[0]):
      print(f"Sample {j + 1}:")
      x_audio_j = x_audio[j].cpu()
      y_audio_j = y_audio[j].cpu()

      display(Audio(x_audio_j.numpy(), rate=sampling_rate))
      display(Audio(y_audio_j.numpy(), rate=sampling_rate))


def compute_losses(model, loader, n_batches=1000):
  model.eval()
  model = model.cuda()
  total_recons_loss = 0.0
  total_kld = 0.0
  total_samples = 0
  with torch.no_grad():
    for i, batch in enumerate(loader):
      batch = batch.to('cuda')
      _, losses = model._autoencode(batch)
      recons_loss = losses['recons_loss'].item()
      kld = losses['kld'].item()
      total_recons_loss += recons_loss * batch.size(0)
      total_kld += kld * batch.size(0)
      total_samples += batch.size(0)
      if i + 1 >= n_batches:
        break
  mean_recons_loss = total_recons_loss / total_samples
  mean_kld = total_kld / total_samples
  return mean_recons_loss, mean_kld


## Train model A

In [ ]:
model_A, trainer_A, train_loader_A, val_loader_A = train_on_dataset('/mnt/mariadata/datasets/syntex/DS_Pistons_1.1/audio', model_name=f'model_A_beta_{beta}')

### Results

In [ ]:
examine_model(model_A, val_loader_A)

## Train model B

In [ ]:
model_B, trainer_B, train_loader_B, val_loader_B = train_on_dataset('/mnt/mariadata/datasets/syntex/DS_BasicFM_1.1/audio', model_name=f'model_B_beta_{beta}')

In [ ]:
examine_model(model_B, val_loader_B)

### Results

## Train model A+B

In [ ]:
model_AB, trained_AB, train_loader_AB, val_loader_AB = train_on_dataset('/mnt/mariadata/datasets/syntex/DS_*/audio', model_name=f'model_AB_beta_{beta}')

In [ ]:
examine_model(model_AB, val_loader_AB)

### Mean loss per dataset

In [ ]:
mrstft_AB_A, kld_AB_A = compute_losses(model_AB, val_loader_A) # losses for model AB on dataset A
mrstft_AB_B, kld_AB_B = compute_losses(model_AB, val_loader_B) # losses for model AB on dataset B

### Plotting latent space

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
import torch
import seaborn as sns

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Helper to encode a batch into latent space
def encode_latents(model, loader, n_batches=2):
  model.eval()
  model.streaming(False)
  model = model.cuda()
  latents = []
  with torch.no_grad():
    for i, batch in enumerate(loader):
      batch = batch.to('cuda')
      # Get mean of latent distribution (mu)
      mu, logvar = model.encoder(batch)
      z, _ = model.encoder.reparametrize(mu, logvar)
      # z = logvar
      # z = mu
      latents.append(z.cpu())
      if i + 1 >= n_batches:
        break
  return torch.concatenate(latents, axis=0)

# Encode samples
latents_A = encode_latents(model_AB, val_loader_A)
latents_B = encode_latents(model_AB, val_loader_B)

# Bin to bin latents
latents_A_bins = latents_A.reshape(-1, latents_A.shape[-1])
latents_B_bins = latents_B.reshape(-1, latents_B.shape[-1])

# Stack and create labels
X = np.vstack([latents_A_bins, latents_B_bins])
y = np.array(['A'] * len(latents_A_bins) + ['B'] * len(latents_B_bins))

# UMAP to 2D
umap_model = umap.UMAP(n_components=2, random_state=42)
X_umap = umap_model.fit_transform(X)

In [ ]:
# Higher resolution for better readability
fig, ax = plt.subplots(figsize=(10, 8), dpi=150)

for label, color in zip(['A', 'B'], ['#1f77b4', '#ff7f0e']):
  idx = y == label
  sns.scatterplot(
    x=X_umap[idx, 0], y=X_umap[idx, 1],
    label=f'Dataset {label}', alpha=0.5, s=2, marker='o', color=color, ax=ax, edgecolor=None
  )

ax.set_title(f'z (UMAP): Dataset A vs B, β={beta}', fontsize=14, pad=25)

subtitle = (
  f"KLD (A): {kld_AB_A:.3f}, KLD (B): {kld_AB_B:.3f}, "
  f"recons_loss (A): {mrstft_AB_A:.3f}, recons_loss (B): {mrstft_AB_B:.3f}"
)

ax.text(0.5, 1.01, subtitle, fontsize=11, ha='center', va='bottom', transform=ax.transAxes)

# Remove axes
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

# Add Legend
# Custom handles for larger dots in legend
from matplotlib.lines import Line2D
handles = [
    Line2D([0], [0], marker='o', color='w', label='Dataset A',
           markerfacecolor='#1f77b4', markersize=8),
    Line2D([0], [0], marker='o', color='w', label='Dataset B',
           markerfacecolor='#ff7f0e', markersize=8)
]
ax.legend(handles=handles, scatterpoints=1, loc='best', frameon=True)

# Remove grid
ax.grid(False)

# ax.set_facecolor("white")
# fig.patch.set_facecolor("white")

plt.show()

In [ ]:
latents_A_means = latents_A.mean(dim=1)
latents_B_means = latents_B.mean(dim=1)

# Scatter plot of means
plt.figure(figsize=(8, 6))
plt.scatter(latents_A_means[:, 0], latents_A_means[:, 1], label='Dataset A', alpha=0.5, s=10, color='blue')
plt.scatter(latents_B_means[:, 0], latents_B_means[:, 1], label='Dataset B', alpha=0.5, s=10, color='orange')
plt.xlabel('Latent Dimension 1')
plt.ylabel('Latent Dimension 2')
plt.title('means of μ: Dataset A vs B')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import random

# Pick one random sample from each validation loader
def get_random_sample(loader):
  batch = next(iter(loader))
  idx = random.randint(0, batch.shape[0] - 1)
  return batch[idx:idx+1]  # keep batch dimension

# Get random samples
sample_A = get_random_sample(val_loader_A)
sample_B = get_random_sample(val_loader_B)

# Encode to latent mus using model_AB
model_AB.eval()
with torch.no_grad():
  mu_A, _ = model_AB.encoder(sample_A.to('cuda'))
  mu_B, _ = model_AB.encoder(sample_B.to('cuda'))

# Move to cpu and numpy
mu_A_np = mu_A.squeeze(0).cpu().numpy()  # shape: (bins, latent_size)
mu_B_np = mu_B.squeeze(0).cpu().numpy()

# Stack for PCA
X_samples = np.vstack([mu_A_np, mu_B_np])
labels = np.array(['A'] * mu_A_np.shape[0] + ['B'] * mu_B_np.shape[0])

# PCA to 2D
pca_samples = PCA(n_components=2)
X_samples_pca = pca_samples.fit_transform(X_samples)

# Plot
plt.figure(figsize=(8, 6))
for label, color in zip(['A', 'B'], ['blue', 'orange']):
  idx = labels == label
  plt.scatter(X_samples_pca[idx, 0], X_samples_pca[idx, 1], label=f'Sample {label}', alpha=0.7, s=20, c=color)
plt.xlabel('PC 1')
plt.ylabel('PC 2')
plt.title('μ across individual samples from A and B')
plt.legend()
plt.grid(True)
plt.show()